In [ ]:
from scipy.signal import butter, filtfilt

def butter_bandpass(lowcut, highcut, fs, order=5):

    nyq = 0.5 * fs

    low = lowcut / nyq

    high = highcut / nyq

    b, a = butter(
        order,
        [low, high],
        btype='band'
    )

    return b, a


def apply_bandpass_filter(data, lowcut, highcut, fs):

    b, a = butter_bandpass(
        lowcut,
        highcut,
        fs
    )

    filtered = filtfilt(
        b,
        a,
        data,
        axis=-1
    )

    return filtered

In [ ]:
def subtract_baseline(data, baseline_samples):

    baseline = np.mean(
        data[:, :, :, :baseline_samples],
        axis=-1,
        keepdims=True
    )

    return data - baseline

In [ ]:
def subject_wise_z_score(data):

    normalized = np.zeros_like(data)

    for subj in range(data.shape[0]):

        mean = np.mean(data[subj])

        std = np.std(data[subj])

        normalized[subj] = (
            data[subj] - mean
        ) / std

    return normalized

In [ ]:
from scipy import signal
import matplotlib.pyplot as plt

sample = X_subj[0, 0, 0]

print(sample.shape)

In [ ]:
frequencies, times, Sxx = signal.spectrogram(
    sample,
    fs=128
)

plt.figure(figsize=(10, 4))

plt.pcolormesh(
    times,
    frequencies,
    Sxx,
    shading='gouraud'
)

plt.ylabel('Frequency')

plt.xlabel('Time')

plt.title("EEG Spectrogram")

plt.colorbar()

plt.show()

In [ ]:
from scipy import signal
from tensorflow.image import resize

sample = X_subj[0, 0, 0]

frequencies, times, Sxx = signal.spectrogram(
    sample,
    fs=128
)

Sxx = np.log(Sxx + 1e-8)

img = resize(
    Sxx[..., np.newaxis],
    (128, 128)
)

img = np.array(img)

print(img.shape)

In [ ]:
import tensorflow as tf

print(tf.config.list_physical_devices('GPU'))

In [ ]:
from scipy import signal
from tensorflow.image import resize

def eeg_to_spectrogram(eeg_trial):

    channel_images = []

    for ch in range(eeg_trial.shape[0]):

        frequencies, times, Sxx = signal.spectrogram(
            eeg_trial[ch],
            fs=128
        )

        Sxx = np.log(
            Sxx + 1e-8
        )

        Sxx = resize(
            Sxx[..., np.newaxis],
            (64, 64)
        )

        Sxx = np.array(Sxx).squeeze()

        channel_images.append(Sxx)

    image = np.stack(
        channel_images,
        axis=-1
    )

    return image

In [ ]:
sample_trial = X_subj[0, 0]

img = eeg_to_spectrogram(
    sample_trial
)

print(img.shape)

In [ ]:
plt.figure(figsize=(6,6))

plt.imshow(img[:, :, 0])

plt.title("EEG Spectrogram Channel 1")

plt.colorbar()

plt.show()

In [ ]:
X_images = []
y_images = []

num_subjects = 31
for subj in range(num_subjects):

    print(f"Processing Subject {subj+1}")

    for trial in range(X_subj.shape[1]):

        eeg_trial = X_subj[subj, trial]

        img = eeg_to_spectrogram(
            eeg_trial
        )

        X_images.append(img)

        y_images.append(
            y_bin[subj, trial]
        )

X_images = np.array(
    X_images,
    dtype=np.float32
)

y_images = np.array(
    y_images,
    dtype=np.int32
)

print(X_images.shape)
print(y_images.shape)

In [ ]:
X_images = (
    X_images - np.min(X_images)
) / (
    np.max(X_images) - np.min(X_images)
)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_images,
    y_images,
    test_size=0.2,
    random_state=42,
    stratify=y_images
)

In [ ]:
import tensorflow as tf

from tensorflow.keras.layers import (
    Input,
    Conv2D,
    MaxPooling2D,
    Reshape,
    MultiHeadAttention,
    LayerNormalization,
    Dense,
    GlobalAveragePooling1D,
    Dropout
)

from tensorflow.keras.models import Model

In [ ]:
input_layer = Input(
    shape=(64, 64, 32)
)

x = Conv2D(
    32,
    (3,3),
    activation='relu',
    padding='same'
)(input_layer)

x = MaxPooling2D((2,2))(x)

x = Conv2D(
    64,
    (3,3),
    activation='relu',
    padding='same'
)(x)

x = MaxPooling2D((2,2))(x)

x = Reshape(
    (16*16, 64)
)(x)

attention_output = MultiHeadAttention(
    num_heads=4,
    key_dim=64
)(
    x,
    x
)

x = LayerNormalization()(attention_output)

x = GlobalAveragePooling1D()(x)

x = Dropout(0.5)(x)

x = Dense(
    64,
    activation='relu'
)(x)

output_layer = Dense(
    2,
    activation='softmax'
)(x)

model = Model(
    inputs=input_layer,
    outputs=output_layer
)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(
        X_test,
        y_test
    ),
    epochs=20,
    batch_size=8
)

In [ ]:
plt.plot(history.history['accuracy'])

plt.plot(history.history['val_accuracy'])

plt.title("Model Accuracy")

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.legend(['Train', 'Validation'])

plt.show()

In [ ]:
plt.plot(history.history['loss'])

plt.plot(history.history['val_loss'])

plt.title("Model Loss")

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.legend(['Train', 'Validation'])

plt.show()

In [ ]:
sample = X_test[0]

prediction = model.predict(
    sample[np.newaxis, ...]
)

print(prediction)

In [ ]:
plt.figure(figsize=(8,8))

plt.imshow(
    sample[:, :, 0],
    cmap='inferno'
)

plt.title("EEG Spectrogram")

plt.colorbar()

plt.show()

In [ ]:
sample = X_test[0]

prediction = model.predict(
    sample[np.newaxis, ...]
)

predicted_class = np.argmax(
    prediction
)

print("Prediction Probabilities:")

print(prediction)

print("Predicted Class:")

print(predicted_class)

In [ ]:
plt.figure(figsize=(8,8))

plt.imshow(
    sample[:, :, 0],
    cmap='inferno'
)

plt.title(
    f"EEG Spectrogram | Predicted Class: {predicted_class}"
)

plt.colorbar()

plt.show()

In [ ]:
from tensorflow.keras.models import Model

attention_layer_model = Model(
    inputs=model.input,
    outputs=model.layers[7].output
)

attention_features = attention_layer_model.predict(
    sample[np.newaxis, ...]
)

print(attention_features.shape)

In [ ]:
attention_map = np.mean(
    attention_features[0],
    axis=-1
)

plt.figure(figsize=(10,4))

plt.plot(attention_map)

plt.title("Attention Importance Across Tokens")

plt.xlabel("Token Index")

plt.ylabel("Importance")

plt.show()

In [ ]:
plt.plot(history.history['accuracy'])

plt.plot(history.history['val_accuracy'])

plt.title("Model Accuracy")

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.legend(['Train', 'Validation'])

plt.savefig(
    "/kaggle/working/accuracy_curve.png"
)

plt.show()

In [ ]:
plt.plot(history.history['loss'])

plt.plot(history.history['val_loss'])

plt.title("Model Loss")

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.legend(['Train', 'Validation'])

plt.savefig(
    "/kaggle/working/loss_curve.png"
)

plt.show()

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
grad_model = tf.keras.models.Model(
    [model.inputs],
    [
        model.get_layer(index=3).output,
        model.output
    ]
)

In [ ]:
sample = X_test[0]

input_image = sample[np.newaxis, ...]

with tf.GradientTape() as tape:

    conv_outputs, predictions = grad_model(
        input_image
    )

    class_idx = tf.argmax(
        predictions[0]
    )

    loss = predictions[:, class_idx]

grads = tape.gradient(
    loss,
    conv_outputs
)

pooled_grads = tf.reduce_mean(
    grads,
    axis=(0, 1, 2)
)

conv_outputs = conv_outputs[0]

heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]

heatmap = tf.squeeze(heatmap)

heatmap = np.maximum(
    heatmap,
    0
)

heatmap /= np.max(
    heatmap
)

print(heatmap.shape)

In [ ]:
plt.figure(figsize=(8,8))

plt.imshow(
    sample[:, :, 0],
    cmap='gray'
)

plt.imshow(
    heatmap,
    cmap='jet',
    alpha=0.5
)

plt.title("GradCAM Heatmap")

plt.colorbar()

plt.show()

In [ ]:
channel_names = [

    'Fp1','AF3','F3','F7','FC5','FC1','C3','T7',
    'CP5','CP1','P3','P7','PO3','O1','Oz','Pz',
    'Fp2','AF4','Fz','F4','F8','FC6','FC2','Cz',
    'C4','T8','CP6','CP2','P4','P8','PO4','O2'
]

print(len(channel_names))

In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
info = mne.create_info(
    ch_names=channel_names,
    sfreq=128,
    ch_types='eeg'
)

montage = mne.channels.make_standard_montage(
    'biosemi32'
)

info.set_montage(montage)

In [ ]:
real_importance = np.std(
    sample,
    axis=(0,1)
)

real_importance = (
    real_importance -
    np.min(real_importance)
)

real_importance = (
    real_importance /
    np.max(real_importance)
)

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))

mne.viz.plot_topomap(
    real_importance,
    info,
    cmap='jet',
    contours=6,
    axes=ax,
    show=False
)

plt.title(
    "EEG Channel Importance Topomap"
)

plt.show()